In [ ]:
import os
import json
from PIL import Image
import rasterio
import numpy as np
from tqdm import trange
import torch
from matplotlib import pyplot as plt
from scipy import ndimage
import matplotlib.patches as patches
from samgeo.hq_sam import (
    SamGeo,
    show_image,
    download_file,
    overlay_images,
    tms_to_geotiff,
)
from pathlib import Path
from matplotlib.colors import ListedColormap, BoundaryNorm

In [ ]:
sam = SamGeo(
    model_type="vit_h",
    sam_kwargs={
        "points_per_side": 128,
        "points_per_batch": 32,
        "crop_n_layers": 0,
        
    },
    hq=True,
    device="cuda",
)

In [ ]:
DATASET_ROOT = Path("/data/mburges/datasets/IOD_Datasets/CHAI")
ANNOTATION_FILE = DATASET_ROOT / "annotations" / "instances_train2017.json"
IMAGE_DIR = DATASET_ROOT / "images"

selected_image = 379  # Change this to the desired image ID

# Load COCO annotations
with open(ANNOTATION_FILE, 'r') as f:
    coco_data = json.load(f)


available_img_ids = list({img['id'] for img in coco_data['images']})
print(f"Available image IDs: {len(available_img_ids)}")  # Print first 10 IDs for reference
image_id = available_img_ids[selected_image]

# Create image id to filename mapping
image_id_to_info = {img['id']: img for img in coco_data['images']}

# Create category id to name mapping
category_id_to_name = {cat['id']: cat['name'] for cat in coco_data['categories']}


if image_id not in image_id_to_info:
    raise ValueError(f"Image ID {image_id} not found in dataset")

image_info = image_id_to_info[image_id]
image_path = IMAGE_DIR / image_info['file_name']

# Load image
image = Image.open(image_path).convert("RGB")
image_np = np.array(image)
# Get all annotations for this image
annotations = [ann for ann in coco_data['annotations'] if ann['image_id'] == image_id]
    
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(image_np)

# Draw bounding boxes
for i, ann in enumerate(annotations):
    x, y, width, height = ann['bbox']

    # Create a yellow rectangle with a black outline
    # We'll simulate the outline by drawing a thicker black rectangle underneath
    # First, draw the black outline (thicker rectangle)
    outline_rect = plt.Rectangle((x, y), width, height, fill=False,
                                    edgecolor='black', linewidth=4, alpha=0.7)
    ax.add_patch(outline_rect)
    # Then, draw the yellow rectangle (thinner rectangle on top)
    rect = plt.Rectangle((x, y), width, height, fill=False,
                            edgecolor='yellow', linewidth=2, clip_on=True)
    ax.add_patch(rect)


ax.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)
plt.gca().xaxis.set_major_locator(plt.NullLocator())
plt.gca().yaxis.set_major_locator(plt.NullLocator())

plt.savefig(f"../plots/CHAI_image_{image_id}.png", bbox_inches='tight', pad_inches=0.1)
plt.show()     


In [ ]:
with torch.no_grad() and torch.autocast("cuda"):
    sam.generate(image_np, foreground=True, unique=True)

point_grids = sam.mask_generator.point_grids  # Not used further here, but timed
masks = sam.masks
sorted_masks = sorted(masks, key=lambda x: x["area"], reverse=False)

objects_stacked = np.zeros(
    (
        sorted_masks[0]["segmentation"].shape[0],
        sorted_masks[0]["segmentation"].shape[1],
        len(sorted_masks),
    )
)
boxes = []
for index, ann in enumerate(sorted_masks):
    m = ann["segmentation"]
    boxes.append(ann["bbox"])
    objects_stacked[:, :, index] = m
print(objects_stacked.shape)

seg = objects_stacked.transpose(2, 0, 1)
idx_array = np.arange(seg.shape[0]).reshape(-1, 1, 1) + 1
mask_bool = seg.astype(bool)
objects = np.sum(mask_bool * idx_array, axis=0)
dtype = np.uint16
objects = objects.astype(dtype)

print(objects.shape)

In [ ]:
def make_label_cmap(num_labels, seed=0, saturation=0.65, value=0.95):
    """
    Builds a ListedColormap for labels in [0..num_labels].
    0 maps to white. 1..num_labels get distinct hues.
    """
    rng = np.random.default_rng(seed)

    # Evenly spaced hues, then shuffle to avoid similar neighbors
    hues = np.linspace(0.0, 1.0, num_labels, endpoint=False)
    rng.shuffle(hues)

    # HSV to RGB
    import colorsys
    rgb = np.array([colorsys.hsv_to_rgb(h, saturation, value) for h in hues])

    # Insert white for background at index 0
    colors = np.vstack(([1.0, 1.0, 1.0], rgb))
    return ListedColormap(colors, name="labels_cmap")

In [ ]:
num_labels = int(objects.max())
cmap = make_label_cmap(num_labels=num_labels, seed=42)

# Use a BoundaryNorm so labels are drawn as flat regions
bounds = np.arange(num_labels + 2) - 0.5
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(1, 1, figsize=(12, 12), dpi=100)
ax.imshow(image_np)
im = ax.imshow(objects, cmap=cmap, norm=norm, interpolation="nearest", alpha=0.3)

# Draw bounding boxes
for i, ann in enumerate(annotations):
    x, y, width, height = ann['bbox']

    # Create a yellow rectangle with a black outline
    # We'll simulate the outline by drawing a thicker black rectangle underneath
    # First, draw the black outline (thicker rectangle)
    outline_rect = plt.Rectangle((x, y), width, height, fill=False,
                                    edgecolor='black', linewidth=4, alpha=0.7)
    ax.add_patch(outline_rect)
    # Then, draw the yellow rectangle (thinner rectangle on top)
    rect = plt.Rectangle((x, y), width, height, fill=False,
                            edgecolor='yellow', linewidth=2, clip_on=True)
    ax.add_patch(rect)

ax.set_axis_off()

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)
plt.gca().xaxis.set_major_locator(plt.NullLocator())
plt.gca().yaxis.set_major_locator(plt.NullLocator())

plt.savefig(f"../plots/CHAI_segm_{image_id}.png", bbox_inches='tight', pad_inches=0.1)
plt.show()